# 🛠️ Kernel & Imports Setup

Run this section **once** at the start of every session — it selects the
right Python interpreter, makes the local `pi_basics` package importable,
and verifies the OpenCV / NumPy stack is installed.



## 1. Select the Jupyter kernel

In VS Code, click the kernel picker in the top-right of this notebook
and choose the project's interpreter:

- **Conda users:** `pi-basics` (created via `conda env create -f environment.yml`)
- **Venv users:** `.venv` at the project root
- **Raspberry Pi:** the venv created with `python3 -m venv .venv`

If the kernel does not appear, install `ipykernel` into the environment
and register it:

```bash
pip install ipykernel
python -m ipykernel install --user --name pi-basics --display-name "Python (pi-basics)"
```



## 2. Install dependencies (first run only)

From a terminal at the project root:

```bash
pip install -r requirements.txt
```

To enable the optional ML stack (PyTorch / TensorFlow), uncomment the
marked block at the bottom of `requirements.txt` before installing.



## 3. Run the setup cell below

The cell does three things:

1. Walks up from the current working directory until it finds
   `src/pi_basics`, treating that as the project root.
2. Inserts `<project_root>/src` into `sys.path` so
   `from pi_basics import ...` works without installing the package.
3. Calls `configure_python_runtime(...)`, which prints the Python
   version, platform, missing packages, and Raspberry Pi-specific
   install hints when applicable.

The result is stored in the variable `setup` for later inspection.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in [cwd, *cwd.parents]
        if (candidate / "src" / "pi_basics").exists()
    ),
    cwd,
)
src_dir = project_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from pi_basics import configure_python_runtime

setup = configure_python_runtime(start_path=project_root, verbose=True)
setup

Python: 3.9.25
Platform: Darwin-arm64
Project root: /Volumes/1TB_DAVINCI/VIDEO-features/katch-and-mike-prework-IMPORTANT/pi-basics
Core dependencies are available.


SetupResult(project_root=PosixPath('/Volumes/1TB_DAVINCI/VIDEO-features/katch-and-mike-prework-IMPORTANT/pi-basics'), python_version='3.9.25', platform_name='Darwin-arm64', missing_packages=[], package_versions={'numpy': '2.0.2', 'pandas': '2.3.3', 'scipy': '1.13.1', 'scikit-learn': '1.6.1', 'opencv-python': '4.13.0'}, warnings=['Apple Silicon detected. If binary installs fail, prefer conda-forge packages.'], raspberry_pi_model=None, install_hints=[])

## 4. Standard imports for the tutorials

The cell below imports the libraries used throughout this notebook.
Re-run it after restarting the kernel.


In [2]:
# Core scientific / vision stack
import cv2
import numpy as np
from pathlib import Path

# Notebook display helpers
import ipywidgets.widgets as widgets
from IPython.display import display

def bgr8_to_jpeg(value, quality=75):
    """Encode a BGR ndarray as JPEG bytes for ipywidgets.Image."""
    return bytes(cv2.imencode('.jpg', value)[1])

print(f"cv2     {cv2.__version__}")
print(f"numpy   {np.__version__}")
print("Imports ready.")


cv2     4.13.0
numpy   2.0.2
Imports ready.


# 📚 PDF Learning Checklist

Work through the PDFs in order. Tick each box (replace `[ ]` with `[x]`) as you finish — VS Code renders these as checkboxes in the markdown preview.


## 🟢 Foundations — Image Input
*Folder: `image-manipulation/image-input/`*

The four PDFs in this folder cover the absolute basics of working with
images in OpenCV: what OpenCV is, how to read images from disk, how to
display them in a notebook, how to write them back out, and how image
quality / compression behaves. Each subsection below summarizes one PDF
and provides a runnable code demo.

> The demo cells assume an image named `render1.jpg` and `render2.jpg` exists in the
> current working directory. Replace the path with any local image to
> follow along.


### 1. Introduction to Open Source CV
📄 [1.Introduction to Open Source CV.pdf](image-manipulation/image-input/1.Introduction%20to%20Open%20Source%20CV.pdf)

OpenCV (Open Source Computer Vision Library) is a cross-platform library
for real-time computer vision. The Python bindings are exposed via the
`cv2` module. Quick sanity check — print the installed version and a
handful of useful constants:


In [3]:
"""
Comprehensive sanity check for OpenCV.

Demonstrates every flag enum / capability discovery API a beginner
typically needs to confirm an OpenCV install is healthy.
"""
import cv2

# 1. Version + build info ----------------------------------------------------
print(f"OpenCV version : {cv2.__version__}")
build = cv2.getBuildInformation()
print("Build info (first 8 lines):")
print("\n".join(build.splitlines()[:8]))

# 2. Threading / hardware ----------------------------------------------------
print(f"\nThreads        : {cv2.getNumThreads()}")
print(f"CPUs           : {cv2.getNumberOfCPUs()}")
print(f"Optimized      : {cv2.useOptimized()}")
try:
    print(f"OpenCL avail.  : {cv2.ocl.haveOpenCL()}")
except AttributeError:
    print("OpenCL avail.  : (cv2.ocl not available)")

# 3. Every cv2.IMREAD_* flag (used by cv2.imread) ----------------------------
print("\nAll cv2.IMREAD_* flags:")
for name in sorted(n for n in dir(cv2) if n.startswith("IMREAD_")):
    print(f"  {name:<35} = {getattr(cv2, name)}")

# 4. Every cv2.IMWRITE_* flag (used by cv2.imwrite params) -------------------
print("\nAll cv2.IMWRITE_* flags:")
for name in sorted(n for n in dir(cv2) if n.startswith("IMWRITE_")):
    print(f"  {name:<45} = {getattr(cv2, name)}")

# 5. Every cv2.COLOR_* conversion code (used by cv2.cvtColor) ---------------
color_codes = sorted(n for n in dir(cv2) if n.startswith("COLOR_"))
print(f"\nTotal cv2.COLOR_* conversion codes: {len(color_codes)}")
print("First 10:", color_codes[:10])


OpenCV version : 4.13.0
Build info (first 8 lines):

General configuration for OpenCV 4.13.0 =====================================
  Version control:               4.13.0-1-gb4c5ec4042

  Platform:
    Timestamp:                   2026-02-03T08:00:24Z
    Host:                        Darwin 22.6.0 arm64
    CMake:                       3.31.4

Threads        : 8
CPUs           : 8
Optimized      : True
OpenCL avail.  : True

All cv2.IMREAD_* flags:
  IMREAD_ANYCOLOR                     = 4
  IMREAD_ANYDEPTH                     = 2
  IMREAD_COLOR                        = 1
  IMREAD_COLOR_BGR                    = 1
  IMREAD_COLOR_RGB                    = 256
  IMREAD_GRAYSCALE                    = 0
  IMREAD_IGNORE_ORIENTATION           = 128
  IMREAD_LOAD_GDAL                    = 8
  IMREAD_REDUCED_COLOR_2              = 17
  IMREAD_REDUCED_COLOR_4              = 33
  IMREAD_REDUCED_COLOR_8              = 65
  IMREAD_REDUCED_GRAYSCALE_2          = 16
  IMREAD_REDUCED_GRAYSCALE_4       

### 2. Image reading and display
📄 [2.Image reading and display.pdf](image-manipulation/image-input/2.Image%20reading%20and%20display.pdf)

OpenCV exposes several APIs for getting pixels into and out of memory and
onto the screen. The demo cell below exercises **all** of them; the
table here is a quick reference.

#### `cv2.imread(path, flag)` — load from disk

| Flag | Value | Meaning |
|------|-------|---------|
| `cv2.IMREAD_UNCHANGED` | `-1` | Keep original format (incl. alpha) |
| `cv2.IMREAD_GRAYSCALE` | `0`  | Force single-channel grayscale |
| `cv2.IMREAD_COLOR` | `1` | Force 3-channel BGR (default) |
| `cv2.IMREAD_ANYDEPTH` | `2` | Preserve 16/32-bit depth |
| `cv2.IMREAD_ANYCOLOR` | `4` | Any colour format as-is |
| `cv2.IMREAD_LOAD_GDAL` | `8` | Use GDAL driver (if built) |
| `cv2.IMREAD_REDUCED_GRAYSCALE_2` / `_4` / `_8` | `16` / `32` / `64` | Grayscale, 1/2 / 1/4 / 1/8 size |
| `cv2.IMREAD_REDUCED_COLOR_2` / `_4` / `_8` | `17` / `33` / `65` | Colour, 1/2 / 1/4 / 1/8 size |
| `cv2.IMREAD_IGNORE_ORIENTATION` | `128` | Ignore EXIF rotation |

#### Related read APIs

| Function | Purpose |
|----------|---------|
| `cv2.imreadmulti(path, flags)` | Read multi-page TIFF / animated GIF into a list of frames |
| `cv2.imdecode(buf, flag)` | Decode an image from an in-memory `np.uint8` byte buffer |
| `cv2.haveImageReader(path)` / `cv2.haveImageWriter(path)` | Probe whether the build can read/write a given file |

#### Display options

| Function | Notes |
|----------|-------|
| `cv2.imshow(name, img)` | Open a desktop window — only works in a `.py` script, **not** Jupyter |
| `cv2.namedWindow(name, flags)` | Create a window with `WINDOW_AUTOSIZE`, `WINDOW_NORMAL`, `WINDOW_KEEPRATIO`, `WINDOW_GUI_NORMAL`, `WINDOW_GUI_EXPANDED` |
| `cv2.setWindowProperty(name, prop, value)` | Tweak via `WND_PROP_FULLSCREEN`, `WND_PROP_AUTOSIZE`, `WND_PROP_ASPECT_RATIO`, ... |
| `cv2.waitKey(ms)` / `cv2.waitKeyEx(ms)` / `cv2.pollKey()` | Block / extended-key / non-blocking key polling |
| `cv2.destroyWindow(name)` / `cv2.destroyAllWindows()` | Tear down windows |

Inside Jupyter we cannot use `cv2.imshow`, so the demo encodes each
array to JPEG with `cv2.imencode` and pushes the bytes through an
`ipywidgets.Image` widget instead.



In [6]:
"""
Demonstrate every flag/option of the cv2 image-reading and display APIs.

Covers:
  * cv2.imread        - every IMREAD_* flag (enumerated dynamically)
  * cv2.imreadmulti   - multi-page / animated readers
  * cv2.imdecode      - decode from a bytes buffer / np.ndarray
  * cv2.haveImageReader - format support probe
  * cv2.imshow + cv2.namedWindow - every WINDOW_* + WND_PROP_* flag
  * cv2.waitKey / cv2.waitKeyEx / cv2.pollKey
  * cv2.destroyWindow / cv2.destroyAllWindows
  * notebook-friendly display via ipywidgets.Image
"""
import cv2
import numpy as np
import ipywidgets.widgets as widgets
from IPython.display import display

IMAGE_PATHS = ("render1.jpg", "render2.jpg")  # any local images

# ---------------------------------------------------------------------------
# 1. Enumerate every cv2.IMREAD_* flag dynamically (no missing constants).
# ---------------------------------------------------------------------------
IMREAD_FLAGS = sorted(
    ((name, getattr(cv2, name)) for name in dir(cv2) if name.startswith("IMREAD_")),
    key=lambda nv: nv[1],
)
print("All cv2.IMREAD_* flags:")
for name, value in IMREAD_FLAGS:
    print(f"  {name:<35} = {value}")

# ---------------------------------------------------------------------------
# 2. Enumerate every cv2.WINDOW_* / WND_PROP_* flag for cv2.namedWindow /
#    cv2.setWindowProperty (these are the display-side options).
# ---------------------------------------------------------------------------
WINDOW_FLAGS = sorted(
    (name, getattr(cv2, name)) for name in dir(cv2)
    if name.startswith(("WINDOW_", "WND_PROP_"))
)
print("\nAll cv2.WINDOW_* / WND_PROP_* flags:")
for name, value in WINDOW_FLAGS:
    print(f"  {name:<35} = {value}")

# ---------------------------------------------------------------------------
# 3. Helper: encode a BGR/gray ndarray as JPEG bytes for ipywidgets.Image.
# ---------------------------------------------------------------------------
def bgr8_to_jpeg(value, quality=75):
    """Encode an ndarray as JPEG bytes, suitable for ipywidgets.Image."""
    return bytes(cv2.imencode('.jpg', value, [cv2.IMWRITE_JPEG_QUALITY, quality])[1])

# ---------------------------------------------------------------------------
# 4. Per-image: read with every IMREAD_* flag, display, and exercise the
#    multi-image / decode-from-buffer / format-probe APIs.
# ---------------------------------------------------------------------------
for IMAGE_PATH in IMAGE_PATHS:
    print(f"\n=== {IMAGE_PATH} ===")

    # Format-support probe (returns False for unsupported / missing files).
    print(f"haveImageReader  : {cv2.haveImageReader(IMAGE_PATH)}")
    if hasattr(cv2, "haveImageWriter"):
        print(f"haveImageWriter  : {cv2.haveImageWriter(IMAGE_PATH)}")

    probe = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
    if probe is None:
        print(f"Could not read {IMAGE_PATH!r} - skipping demo.")
        continue

    # 4a. Apply every imread flag and report shape / dtype.
    print(f"\n{'flag':<35} {'value':>5}  shape              dtype")
    print("-" * 70)
    for name, value in IMREAD_FLAGS:
        img = cv2.imread(IMAGE_PATH, value)
        shape = img.shape if img is not None else "(load failed)"
        dtype = img.dtype if img is not None else "-"
        print(f"{name:<35} {value:>5}  {str(shape):<18} {dtype}")

    # 4b. cv2.imreadmulti - multi-page formats (TIFF/GIF). Falls back to a
    #     single-frame list for ordinary JPEG/PNG.
    ok, frames = cv2.imreadmulti(IMAGE_PATH, flags=cv2.IMREAD_UNCHANGED)
    print(f"\nimreadmulti      : ok={ok}, frames={len(frames) if ok else 0}")

    # 4c. cv2.imdecode - decode the file from a raw byte buffer.
    with open(IMAGE_PATH, "rb") as fh:
        buf = np.frombuffer(fh.read(), dtype=np.uint8)
    decoded = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    print(f"imdecode shape   : {None if decoded is None else decoded.shape}")

    # 4d. Notebook-friendly display via ipywidgets.
    image_widget = widgets.Image(format='jpg', width=400, height=400)
    display(image_widget)
    image_widget.value = bgr8_to_jpeg(probe)

# ---------------------------------------------------------------------------
# 5. Desktop window display - reference of every cv2 windowing call.
#     These only work in a regular .py script (not inside Jupyter).
# ---------------------------------------------------------------------------
# WINDOW_FLAG_COMBOS = [
#     cv2.WINDOW_AUTOSIZE,                       # default; not resizable
#     cv2.WINDOW_NORMAL,                         # user-resizable
#     cv2.WINDOW_NORMAL | cv2.WINDOW_KEEPRATIO,  # preserve aspect ratio
#     cv2.WINDOW_GUI_NORMAL,                     # plain Qt window (no toolbar)
#     cv2.WINDOW_GUI_EXPANDED,                   # full Qt window with toolbar
# ]
# for flags in WINDOW_FLAG_COMBOS:
#     cv2.namedWindow("frame", flags)
#     cv2.setWindowProperty("frame", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_NORMAL)
#     cv2.imshow("frame", probe)
#     key = cv2.waitKey(0)        # 0 = wait forever; >0 = milliseconds
#     # cv2.waitKeyEx(0)           # like waitKey but returns extended codes
#     # cv2.pollKey()              # non-blocking variant (returns -1 if none)
#     cv2.destroyWindow("frame")
# cv2.destroyAllWindows()


All cv2.IMREAD_* flags:
  IMREAD_UNCHANGED                    = -1
  IMREAD_GRAYSCALE                    = 0
  IMREAD_COLOR                        = 1
  IMREAD_COLOR_BGR                    = 1
  IMREAD_ANYDEPTH                     = 2
  IMREAD_ANYCOLOR                     = 4
  IMREAD_LOAD_GDAL                    = 8
  IMREAD_REDUCED_GRAYSCALE_2          = 16
  IMREAD_REDUCED_COLOR_2              = 17
  IMREAD_REDUCED_GRAYSCALE_4          = 32
  IMREAD_REDUCED_COLOR_4              = 33
  IMREAD_REDUCED_GRAYSCALE_8          = 64
  IMREAD_REDUCED_COLOR_8              = 65
  IMREAD_IGNORE_ORIENTATION           = 128
  IMREAD_COLOR_RGB                    = 256

All cv2.WINDOW_* / WND_PROP_* flags:
  WINDOW_AUTOSIZE                     = 1
  WINDOW_FREERATIO                    = 256
  WINDOW_FULLSCREEN                   = 1
  WINDOW_GUI_EXPANDED                 = 0
  WINDOW_GUI_NORMAL                   = 16
  WINDOW_KEEPRATIO                    = 0
  WINDOW_NORMAL                       = 0


Image(value=b'', format='jpg', height='400', width='400')


=== render2.jpg ===
haveImageReader  : True
haveImageWriter  : True

flag                                value  shape              dtype
----------------------------------------------------------------------
IMREAD_UNCHANGED                       -1  (4032, 2268, 3)    uint8
IMREAD_GRAYSCALE                        0  (4032, 2268)       uint8
IMREAD_COLOR                            1  (4032, 2268, 3)    uint8
IMREAD_COLOR_BGR                        1  (4032, 2268, 3)    uint8
IMREAD_ANYDEPTH                         2  (4032, 2268)       uint8
IMREAD_ANYCOLOR                         4  (4032, 2268, 3)    uint8
IMREAD_LOAD_GDAL                        8  (4032, 2268, 3)    uint8
IMREAD_REDUCED_GRAYSCALE_2             16  (2016, 1134)       uint8
IMREAD_REDUCED_COLOR_2                 17  (2016, 1134, 3)    uint8
IMREAD_REDUCED_GRAYSCALE_4             32  (1008, 567)        uint8
IMREAD_REDUCED_COLOR_4                 33  (1008, 567, 3)     uint8
IMREAD_REDUCED_GRAYSCALE_8             64  

Image(value=b'', format='jpg', height='400', width='400')

### 3. Picture writing
📄 [3.Picture writing.pdf](image-manipulation/image-input/3.Picture%20writing.pdf)

`cv2.imwrite(path, img)` saves an array back to disk. The output format
is inferred from the file extension (`.jpg`, `.png`, `.bmp`, ...). For
JPEG you can pass `cv2.IMWRITE_JPEG_QUALITY` (0–100); for PNG you can
pass `cv2.IMWRITE_PNG_COMPRESSION` (0–9).


In [5]:
"""
Exercise every cv2.imwrite parameter that the build supports.

cv2.imwrite(path, img, params=None) -> bool
    `params` is a flat list [flag, value, flag, value, ...] picked from
    the cv2.IMWRITE_* enum. Format is inferred from the file extension.
"""
import cv2
from pathlib import Path

IMAGE_PATHS = ("render1.jpg", "render2.jpg")
out_dir = Path("_demo_out")
out_dir.mkdir(exist_ok=True)

for IMAGE_PATH in IMAGE_PATHS:
    print(f"\n=== {IMAGE_PATH} ===")
    img = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
    if img is None:
        print(f"Could not read {IMAGE_PATH!r} — skipping write demo.")
    else:
        # (filename, params) — every common encoder option, gated by attr presence
        # so the cell still runs on builds missing JPEG-XL / AVIF / EXR support.
        JOBS = [
            ("jpeg_q95.jpg",      [cv2.IMWRITE_JPEG_QUALITY, 95]),
            ("jpeg_q30.jpg",      [cv2.IMWRITE_JPEG_QUALITY, 30]),
            ("jpeg_progressive.jpg",
                                  [cv2.IMWRITE_JPEG_QUALITY, 90,
                                   cv2.IMWRITE_JPEG_PROGRESSIVE, 1,
                                   cv2.IMWRITE_JPEG_OPTIMIZE, 1]),
            ("png_c0.png",        [cv2.IMWRITE_PNG_COMPRESSION, 0]),
            ("png_c9.png",        [cv2.IMWRITE_PNG_COMPRESSION, 9,
                                   cv2.IMWRITE_PNG_STRATEGY,
                                   cv2.IMWRITE_PNG_STRATEGY_DEFAULT]),
            ("webp_q80.webp",     [cv2.IMWRITE_WEBP_QUALITY, 80]),
            ("tiff.tiff",         [cv2.IMWRITE_TIFF_COMPRESSION, 1]),
            ("pxm_binary.ppm",    [cv2.IMWRITE_PXM_BINARY, 1]),
            ("bmp.bmp",           []),
        ]

        # Optional formats — only attempt if the constant exists in this build.
        if hasattr(cv2, "IMWRITE_AVIF_QUALITY"):
            JOBS.append(("avif_q60.avif", [cv2.IMWRITE_AVIF_QUALITY, 60]))
        if hasattr(cv2, "IMWRITE_JPEG2000_COMPRESSION_X1000"):
            JOBS.append(("jp2.jp2",
                         [cv2.IMWRITE_JPEG2000_COMPRESSION_X1000, 1000]))
        if hasattr(cv2, "IMWRITE_EXR_TYPE"):
            JOBS.append(("exr.exr",
                         [cv2.IMWRITE_EXR_TYPE, cv2.IMWRITE_EXR_TYPE_HALF]))

        print(f"{'file':<25} {'ok':<4} {'bytes':>10}")
        print("-" * 42)
        for name, params in JOBS:
            out = out_dir / name
            ok = cv2.imwrite(str(out), img, params)
            size = out.stat().st_size if out.exists() else 0
            print(f"{name:<25} {str(ok):<4} {size:>10,}")

        # In-memory encoding (same params as imwrite) ---------------------------
        ok, buf = cv2.imencode(".png", img, [cv2.IMWRITE_PNG_COMPRESSION, 9])
        print(f"\ncv2.imencode('.png', ...) -> ok={ok}, {buf.nbytes:,} bytes")



=== render1.jpg ===
file                      ok        bytes
------------------------------------------
jpeg_q95.jpg              True  2,372,856
jpeg_q30.jpg              True    440,683
jpeg_progressive.jpg      True  1,696,413
png_c0.png                True 27,482,046
png_c9.png                True  8,676,271
webp_q80.webp             True    675,146
tiff.tiff                 True 27,458,078
pxm_binary.ppm            True 27,433,745
bmp.bmp                   True 27,433,782
avif_q60.avif             True    493,379
jp2.jp2                   True 10,768,458


[ WARN:0@18.180] global grfmt_exr.cpp:102 initOpenEXR imgcodecs: OpenEXR codec is disabled. You can enable it via 'OPENCV_IO_ENABLE_OPENEXR' option. Refer for details and cautions here: https://github.com/opencv/opencv/issues/21326


error: OpenCV(4.13.0) /Users/xperience/GHA-OpenCV-Python/_work/opencv-python/opencv-python/opencv/modules/imgcodecs/src/grfmt_exr.cpp:103: error: (-213:The function/feature is not implemented) imgcodecs: OpenEXR codec is disabled. You can enable it via 'OPENCV_IO_ENABLE_OPENEXR' option. Refer for details and cautions here: https://github.com/opencv/opencv/issues/21326 in function 'initOpenEXR'


### 4. Image Quality
📄 [4.Image Quality.pdf](image-manipulation/image-input/4.Image%20Quality.pdf)

Image quality is a trade-off between **resolution** (pixel count),
**bit-depth**, and **compression**. Re-encoding the same picture at
different JPEG qualities makes the trade-off visible — file size shrinks
quickly while perceptual quality degrades more slowly.


In [ ]:
"""
Visualise every dimension of "image quality":

  * spatial resolution     -> cv2.resize() with each interpolation mode
  * bit depth               -> astype to uint8 / uint16 / float32
  * lossy compression       -> JPEG quality sweep + WebP
  * colour subsampling      -> JPEG chroma quality (when available)
"""
import cv2
import numpy as np
from pathlib import Path

IMAGE_PATHS = ("render1.jpg", "render2.jpg")
out_dir = Path("_demo_out")
out_dir.mkdir(exist_ok=True)

for IMAGE_PATH in IMAGE_PATHS:
    print(f"\n=== {IMAGE_PATH} ===")
    img = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
    if img is None:
        print(f"Could not read {IMAGE_PATH!r} — skipping quality demo.")
    else:
        h, w = img.shape[:2]
        print(f"Original: {w}x{h}, channels={img.shape[2]}, dtype={img.dtype}")

        # 1. Resolution sweep with every cv2 interpolation flag -----------------
        INTERPOLATIONS = [
            ("INTER_NEAREST",      cv2.INTER_NEAREST),
            ("INTER_LINEAR",       cv2.INTER_LINEAR),
            ("INTER_AREA",         cv2.INTER_AREA),
            ("INTER_CUBIC",        cv2.INTER_CUBIC),
            ("INTER_LANCZOS4",     cv2.INTER_LANCZOS4),
            ("INTER_LINEAR_EXACT", getattr(cv2, "INTER_LINEAR_EXACT", cv2.INTER_LINEAR)),
            ("INTER_NEAREST_EXACT", getattr(cv2, "INTER_NEAREST_EXACT", cv2.INTER_NEAREST)),
        ]
        print("\nDownscale to 25% with each interpolation:")
        for name, flag in INTERPOLATIONS:
            small = cv2.resize(img, (w // 4, h // 4), interpolation=flag)
            print(f"  {name:<22} -> {small.shape}")

        # 2. Bit-depth conversions ---------------------------------------------
        print("\nBit-depth conversions:")
        for dtype, scale in [(np.uint8, 1), (np.uint16, 257), (np.float32, 1/255.0)]:
            cast = (img.astype(np.float64) * scale).astype(dtype)
            print(f"  {str(dtype):<25} dtype={cast.dtype}, range=({cast.min()}, {cast.max()})")

        # 3. JPEG quality sweep ------------------------------------------------
        print(f"\n{'quality':>8} | {'jpeg bytes':>12}")
        print("-" * 25)
        for q in (100, 95, 75, 50, 25, 10, 1):
            out = out_dir / f"q{q:03d}.jpg"
            cv2.imwrite(str(out), img, [cv2.IMWRITE_JPEG_QUALITY, q])
            print(f"{q:>8} | {out.stat().st_size:>12,}")

        # 4. Optional chroma-quality / luma-quality knobs ----------------------
        extras = {}
        for attr in ("IMWRITE_JPEG_LUMA_QUALITY", "IMWRITE_JPEG_CHROMA_QUALITY"):
            if hasattr(cv2, attr):
                extras[attr] = getattr(cv2, attr)
        if extras:
            params = [cv2.IMWRITE_JPEG_QUALITY, 75]
            for name, flag in extras.items():
                params += [flag, 60]
            out = out_dir / "jpeg_chroma.jpg"
            cv2.imwrite(str(out), img, params)
            print(f"\nJPEG with {list(extras)}: {out.stat().st_size:,} bytes")

        # 5. WebP quality sweep ------------------------------------------------
        print(f"\n{'webp q':>8} | {'bytes':>10}")
        print("-" * 23)
        for q in (100, 75, 50, 25):
            out = out_dir / f"webp_q{q:03d}.webp"
            cv2.imwrite(str(out), img, [cv2.IMWRITE_WEBP_QUALITY, q])
            print(f"{q:>8} | {out.stat().st_size:>10,}")



## 🟡 Pixel-Level Processing
*Folder: `image-manipulation/processing/`*

- [ ] **5.** [Pixel operations](image-manipulation/processing/5.Pixel%20operations.pdf) — Read/modify individual pixel values via NumPy arrays.
- [ ] **13.** [Grayscale processing](image-manipulation/processing/13.Grayscale%20processing.pdf) — Convert color → grayscale and intensity workflows.
- [ ] **14.** [Binary image](image-manipulation/processing/14.Binary%20image.pdf) — Thresholding to produce binary (B/W) images.
- [ ] **15.** [Edge green detection](image-manipulation/processing/15.Edge%20green%20detection.pdf) — Edge-detection algorithms (Canny, Sobel, etc.).

## 🟠 Geometric Transformations
*Folder: `image-manipulation/image-picture-xy-functions/`*

- [ ] **6.** [Image zoom](image-manipulation/image-picture-xy-functions/6.Image%20zoom.pdf) — Resize / scale with interpolation.
- [ ] **7.** [Picture cut](image-manipulation/image-picture-xy-functions/7.Picture%20cut.pdf) — Crop regions of interest.
- [ ] **8.** [Picture pan](image-manipulation/image-picture-xy-functions/8.Picture%20pan.pdf) — Translate (shift) images.
- [ ] **9.** [Picture mirroring](image-manipulation/image-picture-xy-functions/9.Picture%20mirroring.pdf) — Horizontal / vertical flips.
- [ ] **10.** [Affine transformation](image-manipulation/image-picture-xy-functions/10.Affine%20transformation.pdf) — Combined rotation, scale, shear.
- [ ] **11.** [Picture rotation](image-manipulation/image-picture-xy-functions/11.Picture%20rotation.pdf) — Rotate around a center point.
- [ ] **12.** [Perspective transformation](image-manipulation/image-picture-xy-functions/12.perspective%20transformation.pdf) — 4-point perspective warp.

## 🔵 Drawing on Images
*Folder: `image-manipulation/drawing/`*

- [ ] **16.** [Line segment drawing](image-manipulation/drawing/16.Line%20segment%20drawing.pdf) — `cv2.line`.
- [ ] **17.** [Rectangular & circle drawing](image-manipulation/drawing/17.Rectangular%20circle%20drawing.pdf) — `cv2.rectangle`, `cv2.circle`.
- [ ] **18.** [Text and picture drawing](image-manipulation/drawing/18.Text%20and%20picture%20drawing.pdf) — Annotate with `cv2.putText` and overlays.

## 📘 Reference Material (read alongside)

- [ ] [Bash Cheat Sheet](Bash_Cheat_Sheet.pdf) — Shell commands for navigating the Pi.
- [ ] [Working with Hugging Face — cheatsheet](working-with-hugging-face-cheatsheet.pdf) — Pretrained models & pipelines.
- [ ] [Deep Learning with PyTorch (Part 1)](Deep_Learning_with_PyTorch_1.pdf) — Tensors, autograd, first neural networks.

---

### Suggested learning path

1. Foundations (1 → 4)
2. Pixel processing (5, 13 → 15)
3. Geometric transforms (6 → 12)
4. Drawing primitives (16 → 18)
5. Reference cheatsheets as needed; tackle PyTorch last.
